# 05 · Validate-or-plan + honest hit-rate + failure forensics + active-learning loop

**Standard slot:** *validation plan.* **For Project 25 this closes the DBTL loop (D4/D5):**
execute *or fully plan* experimental validation of the top campaign designs (with controls), integrate
any experimental labels back into the success predictor, deliver an **honest hit-rate + failure-
forensics** analysis, a cohort-wide **"lessons learned"** synthesis, and design an **active-learning
loop** (which design to test next) `[stretch]`.

> **No fabricated experimental results.** A design is a *hypothesis* until measured. Any labels you
> integrate must be REAL wet-lab outcomes; absent those, the analysis runs on `EXAMPLE_DATA` and is
> clearly labeled. **Synthesis screening + institutional biosafety/ethics approval are required for any
> real wet-lab work** (see Responsible Research).

Run `00`–`04` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · The experimental validation plan (controls are mandatory)

Write the costed, controlled plan for the **top candidates** from notebook 03/04. The assay depends on
your `design_type`: binders → **SPR/BLI** (K_D/kinetics) + a competition assay; enzymes → an **activity
assay** (kcat/KM) + a dead-mutant control; antibodies → binding + developability. The three controls
are non-negotiable.

In [ ]:
DESIGN_TYPE = "binder"   # your chosen type
PLAN = {
    "binder":   dict(assay="SPR or BLI vs immobilized target (K_D, kon/koff) + competition assay",
                     expression="E. coli BL21(DE3), His-tagged, 16-18 C overnight; target reagent often mammalian/commercial"),
    "antibody": dict(assay="SPR/BLI binding + developability (SEC, Tm/DSF, poly-specificity)",
                     expression="mammalian (Expi293) or yeast display for screening"),
    "enzyme":   dict(assay="activity assay (kcat/KM) on the substrate + catalytic dead-mutant control",
                     expression="E. coli; purify; confirm fold by CD/SEC before activity"),
    "monomer":  dict(assay="go/no-go: express -> SDS-PAGE -> SEC; fold by CD/DSF",
                     expression="E. coli BL21(DE3), 16-18 C overnight"),
}[DESIGN_TYPE]

CONTROLS = {
    "positive": "a known-good binder/enzyme/natural protein — confirms the assay + reagent are active",
    "negative_scrambled": "YOUR OWN top design with the interface scrambled / catalytic residue mutated — must LOSE activity",
    "negative_unrelated": "an unrelated protein of similar size that should not bind/act",
}
print("design_type:", DESIGN_TYPE)
print("assay      :", PLAN["assay"])
print("expression :", PLAN["expression"])
print("controls (mandatory):")
for k, v in CONTROLS.items():
    print(f"  {k}: {v}")
print("\nCosted reagent list + timeline go in the written plan (D4). Synthesis via an IGSC-screening provider.")

## 2 · Integrate experimental labels (when you have them) — no fabrication

When real wet-lab outcomes come back, put them in a small CSV (`design_id, success`) and re-train the
predictor with them via `build_cohort_table(experimental_labels=...)`. The touched rows get
`label_origin="experimental"`. Below we DEMONSTRATE the wiring on an **EXAMPLE_DATA** label set (a
deterministic synthetic stand-in) — on a real run, replace it with your measured outcomes. We NEVER
fabricate K_D/kcat values.

In [ ]:
import ml_predictor as ml
import pandas as pd

cohort = pd.read_csv("results/cohort_table.csv") if __import__("os").path.exists("results/cohort_table.csv")     else ml.build_cohort_table(seed=0)

# EXAMPLE_DATA stand-in for "experimental labels just came back for a handful of designs".
# On a real run: exp = pd.read_csv("data/experimental_labels.csv")  # columns: design_id, success
exp_demo = cohort.sample(min(15, len(cohort)), random_state=0)[["design_id"]].copy()
exp_demo["success"] = (ml.build_cohort_table(seed=7)["success"].head(len(exp_demo)).values)  # synthetic placeholder
exp_demo["NOTE"] = "EXAMPLE_DATA — replace with REAL measured outcomes; never fabricate"

relabeled = ml._attach_experimental(cohort.copy(), exp_demo[["design_id", "success"]])
print("rows now flagged experimental (EXAMPLE_DATA demo):",
      int((relabeled["label_origin"] == "experimental").sum()))
X, y, cols = ml.features_and_label(relabeled)
b2 = ml.train_success_predictor(X, y, feature_names=cols, model="logreg", seed=0)
print(f"re-trained CV-AUC with integrated labels = {b2['cv_auc_mean']:.3f} +/- {b2['cv_auc_std']:.3f} (EXAMPLE_DATA)")
print("On a real run this is where the loop CLOSES: measured outcomes improve the shared filter.")

## 3 · Honest hit-rate + failure forensics

Report the classical-filter hit rate (notebook 03) AND, where labels exist, the **true** success rate
among the designs you actually tested. Then do **failure forensics**: of the designs that passed the
in-silico filter but FAILED experimentally (false positives), what do they have in common? This is the
single most useful output for the next cohort — it tells them which "confident" designs to distrust.

In [ ]:
import numpy as np
# Among labelled designs, contrast feature distributions of experimental success vs failure.
lab = relabeled.copy()
lab = lab[lab["label_origin"] == "experimental"] if (lab["label_origin"] == "experimental").any() else relabeled
print("failure-forensics: mean feature values by outcome (EXAMPLE_DATA):")
forensics = lab.groupby("success")[ml.FEATURE_COLUMNS].mean(numeric_only=True).round(2)
print(forensics.T.rename(columns={0: "FAIL_mean", 1: "SUCCESS_mean"}).to_string())

# "False positives": passed the classical scRMSD+pae filter but labelled failure.
fp_mask = (lab["scrmsd"] <= 2.0) & (lab["pae_interaction"] <= 10.0) & (lab["success"] == 0)
print(f"\n'confident-but-failed' designs (scrmsd<=2 & pae<=10 yet success=0): {int(fp_mask.sum())}")
print("These false positives are the failure-forensics target — what do they share? (EXAMPLE_DATA)")

## 4 · Active-learning loop — which design to test next? `[stretch]`

Wet-lab tests are expensive, so spend them where they are most informative. Two classic acquisition
rules on the predictor's `P(success)`:
- **Exploitation:** test the highest-`P(success)` designs (most likely to work).
- **Exploration / uncertainty:** test designs where `P(success) ≈ 0.5` (the model is least sure — each
  label teaches it the most).

A real loop alternates: test a batch → add labels → re-train → re-rank. Below we rank the campaign
designs by both rules so you can pick the next batch. **EXAMPLE_DATA.**

In [ ]:
import os
camp = pd.read_csv("results/campaign_designs.csv") if os.path.exists("results/campaign_designs.csv") else None
bundle = ml.train_success_predictor(*ml.features_and_label(cohort)[:2],
                                    feature_names=ml.features_and_label(cohort)[2], model="logreg", seed=0)
if camp is not None and bundle["estimator"] is not None:
    cols = bundle["feature_names"]
    use = camp.dropna(subset=[c for c in cols if c in camp.columns]).copy()
    have = [c for c in cols if c in use.columns]
    if len(have) == len(cols) and len(use):
        proba = bundle["estimator"].predict_proba(use[cols].to_numpy(float))[:, 1]
        use["p_success"] = proba
        use["uncertainty"] = 1.0 - (use["p_success"] - 0.5).abs() * 2.0   # 1 at p=0.5, 0 at p=0/1
        exploit = use.sort_values("p_success", ascending=False).head(5)
        explore = use.sort_values("uncertainty", ascending=False).head(5)
        print("ACTIVE LEARNING — next batch to test (EXAMPLE_DATA):")
        print("\n exploitation (highest P(success)):")
        print(exploit[["design_id", "p_success"]].to_string(index=False))
        print("\n exploration (most uncertain, P~0.5):")
        print(explore[["design_id", "p_success", "uncertainty"]].to_string(index=False))
        use.to_csv("results/p25_active_learning_ranking.csv", index=False)
        print("\nsaved results/p25_active_learning_ranking.csv")
    else:
        print("Active-learning scaffold: campaign features incomplete for the model's columns.")
else:
    print("Active-learning scaffold — needs results/campaign_designs.csv + a trained model.")

## 5 · Cohort-wide "lessons learned" synthesis (D5)

The capstone's final synthesis (write it up in the thesis, seed it here):
- **Which features actually predicted success** across the cohort (notebook 04 importances) — and
  which "trusted" single metrics were weak.
- **The honest hit rate** of the classical filter vs the learned predictor (enrichment + recall).
- **Failure forensics:** the signature of confident-but-wrong designs.
- **A concrete proposal to improve `shared/filtering_pipeline.py`** (new `DEFAULT_CUTOFFS` or a learned
  composite score) — pull-requested back for Projects 01–24's successors.
- **The active-learning recommendation:** the next batch to test.

In [ ]:
LESSONS = """# Capstone lessons-learned (fill from YOUR results; EXAMPLE_DATA placeholders)

1. Best single predictor on the cohort: <feature> (enrichment <x>, recall <y>).
2. Learned composite CV-AUC = <a +/- b>, N=<n>; enrichment <e> vs best single <s> -> <beats/does not>.
3. Confident-but-failed signature (failure forensics): <what false positives share>.
4. Proposed shared-filter change (PR): <new cutoffs / add learned score>; evidence: CV-AUC + N.
5. Active-learning next batch: <design_ids> (exploit) + <design_ids> (explore).
6. Caveats: small/biased N; multi-target; in-silico labels where experimental are missing.
"""
open("results/p25_lessons_learned.md", "w").write(LESSONS)
print("wrote results/p25_lessons_learned.md (template — fill with YOUR results).")
print("\nReminder: a design is a HYPOTHESIS until measured; report the hit rate, not the cherry.")

## D4 / D5 checklist
- [ ] Costed, controlled **validation plan** (assay for your design_type; positive + scrambled + unrelated controls).
- [ ] Any **real experimental labels integrated** (`label_origin="experimental"`); model re-trained — no fabrication.
- [ ] **Honest hit-rate** (classical filter vs learned predictor) + **failure forensics** (false-positive signature).
- [ ] **Active-learning** next-batch ranking (exploit + explore) `[stretch]`.
- [ ] Cohort-wide **lessons-learned** synthesis + a concrete `filtering_pipeline.py` improvement proposal.
- [ ] D★: the improved success-predictor module (`scripts/ml_predictor.py`) + honest hit-rate + lessons.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — and the cohort's shared filter is now better because of your learned predictor.